# GPTCloneBench pretrained classification from the embedding notebook output

This notebook consumes only the saved Kaggle Output of
`function-embeddings-for-code-clone-benchmarks`. That output already bundles
the validated clean split, endpoint metadata, and all four embedding families;
the original GPTCloneBench Dataset must not be attached to this notebook.


## 1. Runtime requirements

1. Run and **Save Version** of
   `function-embeddings-for-code-clone-benchmarks` successfully.
2. Choose **Add Input → Notebook Output** and attach only that saved version.
3. Enable a T4-class GPU and select **Run All**. No Dataset, API secret, path,
   or username edit is required.
4. Save this notebook version to retain all paper tables, metrics, predictions,
   and manifests from `/kaggle/working/gptclonebench-classification-results`.

Dataset publication is intentionally disabled, eliminating the Kaggle status
API permission failure while preserving every result as notebook output.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED_KAGGLE_CLI_VERSION = "2.2.3"
REQUIRED_KAGGLE_SDK_VERSION = "0.1.31"

install_specs = []

def installed_version(package_name: str) -> str:
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"

if installed_version("kaggle") != REQUIRED_KAGGLE_CLI_VERSION:
    install_specs.append(f"kaggle=={REQUIRED_KAGGLE_CLI_VERSION}")
if installed_version("kagglesdk") != REQUIRED_KAGGLE_SDK_VERSION:
    install_specs.append(f"kagglesdk=={REQUIRED_KAGGLE_SDK_VERSION}")

for module_name, package_spec in (
    ("numpy", "numpy>=1.26,<3"),
    ("pandas", "pandas>=2.2,<3"),
    ("pyarrow", "pyarrow>=15,<30"),
    ("duckdb", "duckdb>=1.3,<2"),
    ("sklearn", "scikit-learn>=1.4,<2"),
    ("matplotlib", "matplotlib>=3.8,<4"),
    ("joblib", "joblib>=1.3,<2"),
    ("tqdm", "tqdm>=4.66,<5"),
    ("PIL", "Pillow>=10,<13"),
):
    if importlib.util.find_spec(module_name) is None:
        install_specs.append(package_spec)

if install_specs:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade-strategy",
            "only-if-needed",
            *sorted(set(install_specs)),
        ]
    )

print("Dependencies are ready.")
for package in (
    "numpy", "pandas", "pyarrow", "duckdb", "scikit-learn",
"matplotlib", "joblib", "kaggle", "kagglesdk",
):
    print(f"{package}: {installed_version(package)}")

## 2. Pretrained-baseline configuration


In [ ]:
from pathlib import Path

KAGGLE_OWNER = "kousha.m"
SOURCE_EMBEDDING_NOTEBOOK = "function-embeddings-for-code-clone-benchmarks"
EXPECTED_EMBEDDING_PIPELINE_VERSION = "3.0.0-gptclonebench-clean"
SOURCE_DATA_FORMAT = "spectral_clean_data_v1"
EXPECTED_BENCHMARK_CONTENT_VERSION = SOURCE_DATA_FORMAT

OUTPUT_DATASET_SLUG = "gptclonebench-classification-results"
OUTPUT_DATASET_TITLE = "GPTCloneBench Classification Results"
OUTPUT_DATASET_SUBTITLE = "Pretrained baselines on the no-self-pair clean split"
EXPERIMENT_VERSION = "4.0.0-no-self-pair"
OUTPUT_VERSION_NOTES = (
    "CodeBERT-family baselines on GPTCloneBench V3 clean no-self-pair splits"
)

RANDOM_SEED = 42
SOURCE_BENCHMARK_NAME = "GPTCloneBench"
SOURCE_NOTE = "GPTCloneBench V3 clean-data split with all self-pairs removed"
PAPER_DATASET_NAME = "GPTCloneBench"
PAIR_LEFT_ID_COLUMN = "function_id_1"
PAIR_RIGHT_ID_COLUMN = "function_id_2"
SUBGROUP_COLUMN = "language"

EMBEDDING_MODELS_TO_RUN = ["CodeBERT", "GraphCodeBERT", "UniXCode", "CodeT5"]
MODEL_DISPLAY_NAME = {
    "CodeBERT": "CodeBERT", "GraphCodeBERT": "GraphCodeBERT",
    "UniXCode": "UniXcoder", "CodeT5": "CodeT5",
}
BASELINE_VARIANTS = ["No Train", "RF", "SNN", "PCA + RF", "PCA + SNN"]

RF_SEARCH_ITERATIONS = 3
RF_TRAIN_ROW_CAP = None
N_JOBS = 4
SNN_MAX_EPOCHS = 10
SNN_PATIENCE = 3
SNN_BATCH_SIZE = 8192
SNN_HIDDEN_DIM = 256
SNN_EMBED_DIM = 128
SNN_DROPOUT = 0.10
SNN_LEARNING_RATE = 1e-3
SNN_WEIGHT_DECAY = 1e-4
SNN_SCHEDULER_FACTOR = 0.5
SNN_SCHEDULER_PATIENCE = 3
SNN_GRAD_CLIP_NORM = 5.0
SNN_USE_AMP = True
SNN_USE_POS_WEIGHT = False
SNN_TRAIN_ROW_CAP = None

FEATURE_DIMENSION = 768
PAIR_FEATURE_DIMENSION = 1536
PCA_COMPONENTS_PER_ENDPOINT = 32
PCA_PAIR_FEATURE_DIMENSION = 64
FEATURE_MATERIALIZE_BATCH_SIZE = 20_000
PREDICTION_BATCH_SIZE = 20_000
SAVE_FITTED_MODELS = False
SAVE_TEST_PREDICTIONS = True
RESUME_COMPLETED = True
QUICK_TEST = False
QUICK_TEST_ROWS_PER_STRATUM = 300
QUICK_TEST_ROWS_PER_LANGUAGE_LABEL_SPLIT = QUICK_TEST_ROWS_PER_STRATUM
RESET_EXPERIMENT = True
PUBLISH_TO_KAGGLE = False
REQUIRE_EXISTING_OUTPUT_DATASET = False
OUTPUT_DATASET_VISIBILITY = "private"

WORK_ROOT = Path("/kaggle/temp/gptclonebench_pretrained_baselines_clean")
FEATURE_ROOT = WORK_ROOT / "features"
OUTPUT_ROOT = Path("/kaggle/working/gptclonebench-classification-results")

if QUICK_TEST:
    EMBEDDING_MODELS_TO_RUN = ["CodeBERT"]
    RF_SEARCH_ITERATIONS = 2
    RF_TRAIN_ROW_CAP = 2_000
    SNN_TRAIN_ROW_CAP = 2_000
    SNN_MAX_EPOCHS = 2
    SNN_PATIENCE = 1

print("Experiment version:", EXPERIMENT_VERSION)
print("Only input notebook output:", SOURCE_EMBEDDING_NOTEBOOK)
print("Embedding models:", EMBEDDING_MODELS_TO_RUN)
print("Baseline variants:", BASELINE_VARIANTS)


## 3. Imports, reproducibility, and Kaggle authentication

In [ ]:
from __future__ import annotations

import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import duckdb
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image, ImageDraw, ImageFont
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=ConvergenceWarning)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if RESET_EXPERIMENT:
    shutil.rmtree(WORK_ROOT, ignore_errors=True)
    shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def get_kaggle_secret(name: str) -> Optional[str]:
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        value = os.environ.get(name)
        return value.strip() if value else None

def configure_kaggle_auth(required: bool = True) -> str:
    username = KAGGLE_OWNER
    api_token = get_kaggle_secret("KAGGLE_API_TOKEN")
    legacy_key = get_kaggle_secret("KAGGLE_KEY")

    if not username:
        if required:
            raise RuntimeError("KAGGLE_USERNAME is missing.")
        return "missing"

    os.environ["KAGGLE_USERNAME"] = username

    if api_token:
        os.environ["KAGGLE_API_TOKEN"] = api_token
        return "api_token"
    if legacy_key:
        os.environ["KAGGLE_KEY"] = legacy_key
        return "legacy_key"

    if required:
        raise RuntimeError(
            "KAGGLE_API_TOKEN is missing. KAGGLE_KEY is accepted only as a "
            "legacy fallback."
        )
    return "missing"

def run_command(
    command: list[str],
    *,
    check: bool = True,
    quiet: bool = False,
    timeout: int = 600,
) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        timeout=timeout,
        env=os.environ.copy(),
    )
    if not quiet and result.stdout.strip():
        print(result.stdout.strip())
    if check and result.returncode != 0:
        combined = "\n".join(
            part.strip()
            for part in (result.stdout or "", result.stderr or "")
            if part and part.strip()
        )
        raise RuntimeError(
            "Command failed:\n" + " ".join(command) + "\n\n" + combined
        )
    return result

def combined_output(result: subprocess.CompletedProcess[str]) -> str:
    return "\n".join(
        part.strip()
        for part in (result.stdout or "", result.stderr or "")
        if part and part.strip()
    )

AUTH_METHOD = configure_kaggle_auth(required=False)
KAGGLE_USERNAME = KAGGLE_OWNER
OUTPUT_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{OUTPUT_DATASET_SLUG}"

if PUBLISH_TO_KAGGLE:
    if AUTH_METHOD == "missing":
        raise RuntimeError("KAGGLE_API_TOKEN is required only when publication is enabled.")
    run_command(
        ["kaggle", "datasets", "list", "--mine", "--page", "1"],
        quiet=True,
        timeout=180,
    )

print("Authentication method:", AUTH_METHOD)
print("Publication mode:", "Dataset" if PUBLISH_TO_KAGGLE else "saved_notebook_output")


## 4. Locate the first notebook's saved output

The resolver searches only `/kaggle/input` for a complete output produced by
`function-embeddings-for-code-clone-benchmarks`. It does not download or
accept a separately published embedding Dataset. The manifest, validation
report, split files, success markers, and every embedding shard must all belong
to the same discovered output root.


In [ ]:
MODEL_SLUG = {
    "CodeBERT": "codebert", "GraphCodeBERT": "graphcodebert",
    "CodeT5": "codet5", "UniXCode": "unixcode",
}
EXPECTED_SPLIT_ROWS = {"train": 4_144, "validation": 886, "test": 894}
EXPECTED_POSITIVE_ROWS = {"train": 2_072, "validation": 443, "test": 447}
EXPECTED_FULL_PAIR_ROWS = 5_924
EXPECTED_FUNCTION_ROWS = 5_924

def read_json(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected a JSON object in {path}.")
    return value

def candidate_dataset_roots(base: Path) -> list[Path]:
    candidates = [base]
    if base.is_dir():
        candidates.extend(path.parent for path in base.rglob("embedding_manifest.json"))
    unique, seen = [], set()
    for candidate in candidates:
        key = candidate.as_posix()
        if key not in seen:
            seen.add(key); unique.append(candidate)
    return unique

def embedding_root_is_valid(root: Path) -> bool:
    required = [
        root / "embedding_manifest.json", root / "final_validation.json",
        root / "pairs.csv", root / "codes.jsonl", root / "source_metadata.json",
    ]
    if not all(path.is_file() for path in required):
        return False
    try:
        manifest = read_json(root / "embedding_manifest.json")
        final_validation = read_json(root / "final_validation.json")
    except Exception:
        return False
    if (
        str(manifest.get("embedding_pipeline_version")) != EXPECTED_EMBEDDING_PIPELINE_VERSION
        or str(manifest.get("source_data_format")) != SOURCE_DATA_FORMAT
        or int(manifest.get("source_function_counts", {}).get("GPTCloneBench", -1)) != EXPECTED_FUNCTION_ROWS
        or int(manifest.get("self_pair_rows", -1)) != 0
        or final_validation.get("valid") is not True
    ):
        return False
    for model_slug in MODEL_SLUG.values():
        if not (root / f"{model_slug}__gptclonebench__SUCCESS.json").is_file():
            return False
        if not list(root.glob(f"{model_slug}__gptclonebench__part-*.parquet")):
            return False
    return True

def find_valid_dataset_root(base: Path) -> Optional[Path]:
    return next((root for root in candidate_dataset_roots(base) if embedding_root_is_valid(root)), None)

EMBEDDING_ROOT = find_valid_dataset_root(Path("/kaggle/input"))
EMBEDDING_INPUT_METHOD = "attached_notebook_output"
if EMBEDDING_ROOT is None:
    raise FileNotFoundError(
        "No complete embedding notebook output was found under /kaggle/input. "
        "Attach only the saved output of function-embeddings-for-code-clone-benchmarks "
        "through Add Input -> Notebook Output, then run this notebook again."
    )

EMBEDDING_MANIFEST = read_json(EMBEDDING_ROOT / "embedding_manifest.json")
SOURCE_METADATA = read_json(EMBEDDING_ROOT / "source_metadata.json")
print("Embedding root:", EMBEDDING_ROOT)
print("Input method:", EMBEDDING_INPUT_METHOD)
print("Embedding pipeline version:", EMBEDDING_MANIFEST["embedding_pipeline_version"])
print("Bundled benchmark artifacts: PASS")


## 5. Load and validate the bundled no-self-pair split

The exact `pairs.csv` and `codes.jsonl` files come from the embedding
notebook output. This notebook independently rechecks split/class counts,
endpoint coverage, language metadata, absence of self-pairs, and split
isolation before constructing classifier features.


In [ ]:
PAIR_REQUIRED_COLUMNS = {"split", "left_id", "right_id", "label"}
CODE_REQUIRED_COLUMNS = {"code_id", "language"}

full_pairs = pd.read_csv(EMBEDDING_ROOT / "pairs.csv")
codes = pd.read_json(EMBEDDING_ROOT / "codes.jsonl", lines=True)
missing_pair = sorted(PAIR_REQUIRED_COLUMNS - set(full_pairs.columns))
missing_code = sorted(CODE_REQUIRED_COLUMNS - set(codes.columns))
if missing_pair or missing_code:
    raise RuntimeError(f"Missing bundled columns: pairs={missing_pair}, codes={missing_code}")

full_pairs["split"] = full_pairs["split"].astype(str).str.casefold().replace({"valid": "validation"})
raw_left_ids = full_pairs["left_id"].astype(str)
raw_right_ids = full_pairs["right_id"].astype(str)
left_is_first = raw_left_ids.le(raw_right_ids)
full_pairs["function_id_1"] = np.where(left_is_first, raw_left_ids, raw_right_ids)
full_pairs["function_id_2"] = np.where(left_is_first, raw_right_ids, raw_left_ids)
full_pairs["label"] = full_pairs["label"].astype("int8")
full_pairs["pair_id"] = [f"gptclean-{index:06d}" for index in range(len(full_pairs))]
codes["code_id"] = codes["code_id"].astype(str)
codes["language"] = codes["language"].astype(str).str.casefold()

if len(full_pairs) != EXPECTED_FULL_PAIR_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_FULL_PAIR_ROWS:,} pairs, found {len(full_pairs):,}.")
if len(codes) != EXPECTED_FUNCTION_ROWS or codes["code_id"].nunique() != EXPECTED_FUNCTION_ROWS:
    raise RuntimeError("Expected exactly 5,924 unique code rows.")
if full_pairs["function_id_1"].eq(full_pairs["function_id_2"]).any():
    raise RuntimeError("Self-pairs are forbidden in this clean release.")
if set(full_pairs["label"].unique()) != {0, 1}:
    raise RuntimeError("Labels must be exactly {0, 1}.")

observed_rows = full_pairs.groupby("split", observed=True).size().astype(int).to_dict()
observed_positive = full_pairs.loc[full_pairs["label"].eq(1)].groupby("split", observed=True).size().astype(int).to_dict()
if observed_rows != EXPECTED_SPLIT_ROWS:
    raise RuntimeError(f"Split counts changed: {observed_rows}")
if observed_positive != EXPECTED_POSITIVE_ROWS:
    raise RuntimeError(f"Positive counts changed: {observed_positive}")

language_by_id = codes.set_index("code_id")["language"].to_dict()
left_languages = full_pairs["function_id_1"].map(language_by_id)
right_languages = full_pairs["function_id_2"].map(language_by_id)
if left_languages.isna().any() or right_languages.isna().any():
    raise RuntimeError("At least one pair endpoint is absent from codes.jsonl.")
full_pairs["language"] = np.where(
    left_languages.eq(right_languages), left_languages,
    left_languages.astype(str) + "--" + right_languages.astype(str),
)

full_endpoint_ids = set(full_pairs["function_id_1"]) | set(full_pairs["function_id_2"])
if full_endpoint_ids != set(codes["code_id"]):
    raise RuntimeError("Pair endpoints do not exactly match the embedding universe.")
endpoint_rows = pd.concat([
    full_pairs[["split", "function_id_1"]].rename(columns={"function_id_1": "function_id"}),
    full_pairs[["split", "function_id_2"]].rename(columns={"function_id_2": "function_id"}),
], ignore_index=True)
endpoint_cross_split = int(endpoint_rows.groupby("function_id")["split"].nunique().gt(1).sum())
if endpoint_cross_split:
    raise RuntimeError(f"Endpoint leakage across splits: {endpoint_cross_split}")

pairs = full_pairs.copy()
if QUICK_TEST:
    pairs["_quick_order"] = pairs["pair_id"].map(
        lambda value: hashlib.sha256(f"{RANDOM_SEED}|{value}".encode()).hexdigest()
    )
    pairs = (
        pairs.sort_values("_quick_order")
        .groupby(["language", "split", "label"], observed=True, group_keys=False)
        .head(QUICK_TEST_ROWS_PER_LANGUAGE_LABEL_SPLIT)
        .drop(columns="_quick_order")
    )

pairs = pairs.sort_values(["split", "language", "label", "pair_id"]).reset_index(drop=True)
pairs.insert(0, "sample_row_id", np.arange(len(pairs), dtype=np.int64))
PAIR_SELECTION_PATH = OUTPUT_ROOT / "gptclonebench_clean_pairs_used.parquet"
pairs[["sample_row_id", "pair_id", "split", "language", "label", "function_id_1", "function_id_2"]].to_parquet(
    PAIR_SELECTION_PATH, index=False, compression="zstd"
)
split_distribution = pairs.groupby(["split", "language", "label"], observed=True).size().rename("pair_count").reset_index()
split_distribution.to_csv(OUTPUT_ROOT / "split_language_label_distribution.csv", index=False)

input_validation = {
    "status": "PASS", "quick_test": QUICK_TEST,
    "source_data_format": SOURCE_DATA_FORMAT,
    "embedding_pipeline_version": EXPECTED_EMBEDDING_PIPELINE_VERSION,
    "protocol_variant": "clean_no_self_pair",
    "pair_rows": len(pairs), "full_pair_rows": EXPECTED_FULL_PAIR_ROWS,
    "full_function_rows": EXPECTED_FUNCTION_ROWS,
    "full_split_rows": EXPECTED_SPLIT_ROWS,
    "full_positive_rows": EXPECTED_POSITIVE_ROWS,
    "self_pair_rows": 0,
    "endpoint_cross_split_violations": endpoint_cross_split,
}
(OUTPUT_ROOT / "input_validation.json").write_text(json.dumps(input_validation, indent=2, sort_keys=True), encoding="utf-8")
print("GPTCloneBench clean pair validation: PASS")
print("Rows used:", f"{len(pairs):,}")
print("Unique endpoints:", f"{len(full_endpoint_ids):,}")
print("Self-pairs: 0")
display(split_distribution)


## 6. Validate embedding units from the first notebook


In [ ]:
def embedding_shards(model_name: str) -> list[Path]:
    model_slug = MODEL_SLUG[model_name]
    return sorted(
        EMBEDDING_ROOT.glob(
            f"{model_slug}__gptclonebench__part-*.parquet"
        )
    )


def decode_embedding_array(
    array: pa.Array,
    expected_dimension: int,
) -> np.ndarray:
    if pa.types.is_fixed_size_list(array.type):
        dimension = array.type.list_size
        values = array.values.to_numpy(zero_copy_only=False)
        matrix = np.asarray(values).reshape(len(array), dimension)
    elif pa.types.is_list(array.type) or pa.types.is_large_list(array.type):
        offsets = array.offsets.to_numpy(zero_copy_only=False)
        lengths = np.diff(offsets)
        if len(lengths) and not np.all(lengths == expected_dimension):
            raise RuntimeError(
                "Embedding list rows do not have a constant dimension."
            )
        values = array.values.to_numpy(zero_copy_only=False)
        matrix = np.asarray(values).reshape(
            len(array), expected_dimension
        )
    else:
        matrix = np.asarray(array.to_pylist())

    if matrix.shape != (len(array), expected_dimension):
        raise RuntimeError(
            f"Unexpected embedding matrix shape {matrix.shape}; expected "
            f"({len(array)}, {expected_dimension})."
        )
    return matrix.astype(np.float32, copy=False)


required_endpoint_ids = set(
    pd.concat(
        [pairs["function_id_1"], pairs["function_id_2"]],
        ignore_index=True,
    ).astype(str)
)
full_required_endpoint_ids = set(
    pd.concat(
        [
            full_pairs["function_id_1"].astype(str),
            full_pairs["function_id_2"].astype(str),
        ],
        ignore_index=True,
    )
)
if len(full_required_endpoint_ids) != EXPECTED_FUNCTION_ROWS:
    raise RuntimeError(
        "The full GPTCloneBench endpoint set does not contain exactly "
        f"{EXPECTED_FUNCTION_ROWS:,} IDs."
    )

embedding_validation_rows = []
for model_name in EMBEDDING_MODELS_TO_RUN:
    model_slug = MODEL_SLUG[model_name]
    success_path = (
        EMBEDDING_ROOT
        / f"{model_slug}__gptclonebench__SUCCESS.json"
    )
    success = read_json(success_path)

    success_checks = {
        "valid": bool(success.get("valid")),
        "rows": int(success.get("rows", -1)) == EXPECTED_FUNCTION_ROWS,
        "expected_rows": (
            int(success.get("expected_rows", -1)) == EXPECTED_FUNCTION_ROWS
        ),
        "unique_function_ids": (
            int(success.get("unique_function_ids", -1))
            == EXPECTED_FUNCTION_ROWS
        ),
        "dimension": int(success.get("dimension", -1)) == FEATURE_DIMENSION,
        "dataset": str(success.get("dataset")) == "GPTCloneBench",
    }
    failed_success = [
        key for key, value in success_checks.items() if not value
    ]
    if failed_success:
        raise RuntimeError(
            f"{model_name} SUCCESS validation failed: {failed_success}"
        )

    shards = embedding_shards(model_name)
    if not shards:
        raise FileNotFoundError(
            f"No GPTCloneBench embedding shards for {model_name}."
        )

    connection = duckdb.connect()
    try:
        glob_path = str(
            EMBEDDING_ROOT
            / f"{model_slug}__gptclonebench__part-*.parquet"
        ).replace("\\", "/").replace("'", "''")
        relational = connection.execute(
            f"""
            SELECT
                COUNT(*) AS rows,
                COUNT(DISTINCT CAST(function_id AS VARCHAR)) AS unique_ids,
                COUNT(*) FILTER (
                    WHERE CAST(dataset AS VARCHAR) <> 'GPTCloneBench'
                ) AS wrong_dataset_rows,
                COUNT(*) FILTER (WHERE function_id IS NULL) AS null_ids
            FROM read_parquet('{glob_path}')
            """
        ).fetchone()
        id_frame = connection.execute(
            f"""
            SELECT CAST(function_id AS VARCHAR) AS function_id
            FROM read_parquet('{glob_path}')
            """
        ).df()
    finally:
        connection.close()

    rows, unique_ids, wrong_dataset_rows, null_ids = map(int, relational)
    embedding_ids = set(id_frame["function_id"].astype(str))

    checks = {
        "rows": rows == EXPECTED_FUNCTION_ROWS,
        "unique_ids": unique_ids == EXPECTED_FUNCTION_ROWS,
        "wrong_dataset_rows": wrong_dataset_rows == 0,
        "null_ids": null_ids == 0,
        "exact_full_id_match": embedding_ids == full_required_endpoint_ids,
        "all_selected_ids_present": required_endpoint_ids.issubset(
            embedding_ids
        ),
    }
    failed = [key for key, value in checks.items() if not value]
    if failed:
        raise RuntimeError(
            f"{model_name} embedding release mismatch: {failed}"
        )

    embedding_validation_rows.append(
        {
            "embedding_model": model_name,
            "rows": rows,
            "unique_function_ids": unique_ids,
            "shards": len(shards),
            "status": "PASS",
        }
    )

embedding_validation = pd.DataFrame(embedding_validation_rows)
embedding_validation.to_csv(
    OUTPUT_ROOT / "embedding_input_validation.csv",
    index=False,
)
print("GPTCloneBench clean embedding validation: PASS")
display(embedding_validation)

## 7. Load frozen endpoint embeddings

The four transformer encoders are not fine-tuned in this notebook. Their
function embeddings are loaded from the first notebook's output, while RF,
SNN, PCA+RF, and PCA+SNN are trained only on the bundled training pairs.


In [ ]:
from sklearn.decomposition import PCA


def load_endpoint_embeddings(
    model_name: str,
) -> tuple[np.ndarray, list[str], dict[str, int], float]:
    started = time.perf_counter()
    tables = [
        pq.read_table(
            shard,
            columns=["function_id", "dataset", "embedding"],
        )
        for shard in embedding_shards(model_name)
    ]
    table = pa.concat_tables(tables)
    del tables

    ids = [
        str(value)
        for value in table.column("function_id").to_pylist()
    ]
    if len(ids) != len(set(ids)):
        raise RuntimeError(
            f"{model_name}: duplicate function IDs in embeddings."
        )

    embedding_array = table.column("embedding").combine_chunks()
    vectors = decode_embedding_array(
        embedding_array,
        FEATURE_DIMENSION,
    ).astype(np.float32, copy=False)

    if vectors.shape != (len(ids), FEATURE_DIMENSION):
        raise RuntimeError(
            f"{model_name}: unexpected embedding matrix shape "
            f"{vectors.shape}."
        )
    if not np.isfinite(vectors).all():
        raise RuntimeError(
            f"{model_name}: non-finite embedding values."
        )
    if (np.linalg.norm(vectors, axis=1) <= 1e-12).any():
        raise RuntimeError(
            f"{model_name}: zero-norm embedding vector."
        )

    row_by_id = {
        function_id: index
        for index, function_id in enumerate(ids)
    }
    elapsed = time.perf_counter() - started
    del table, embedding_array
    gc.collect()
    return vectors, ids, row_by_id, elapsed


def pair_endpoint_indices(
    row_by_id: dict[str, int],
) -> tuple[np.ndarray, np.ndarray]:
    try:
        left = np.fromiter(
            (
                row_by_id[str(value)]
                for value in pairs[PAIR_LEFT_ID_COLUMN]
            ),
            dtype=np.int32,
            count=len(pairs),
        )
        right = np.fromiter(
            (
                row_by_id[str(value)]
                for value in pairs[PAIR_RIGHT_ID_COLUMN]
            ),
            dtype=np.int32,
            count=len(pairs),
        )
    except KeyError as error:
        raise RuntimeError(
            f"Missing endpoint embedding for ID {error}."
        ) from error
    return left, right


def materialize_pair_features(
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    positions = np.asarray(positions, dtype=np.int64)
    pair_dim = int(vectors.shape[1] * 2)
    matrix = np.empty(
        (len(positions), pair_dim),
        dtype=np.float32,
    )
    for start in range(
        0,
        len(positions),
        FEATURE_MATERIALIZE_BATCH_SIZE,
    ):
        end = min(
            start + FEATURE_MATERIALIZE_BATCH_SIZE,
            len(positions),
        )
        local = positions[start:end]
        dim = vectors.shape[1]
        matrix[start:end, :dim] = vectors[
            left_indices[local]
        ]
        matrix[start:end, dim:] = vectors[
            right_indices[local]
        ]
    if not np.isfinite(matrix).all():
        raise RuntimeError("Non-finite pair features.")
    return matrix


def fit_pca32_from_train_endpoints(
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    train_positions: np.ndarray,
) -> tuple[np.ndarray, PCA, float]:
    started = time.perf_counter()
    train_endpoint_indices = np.unique(
        np.concatenate(
            [
                left_indices[train_positions],
                right_indices[train_positions],
            ]
        )
    )
    if len(train_endpoint_indices) < PCA_COMPONENTS_PER_ENDPOINT:
        raise RuntimeError(
            "Too few training endpoints for 32-component PCA."
        )

    pca = PCA(
        n_components=PCA_COMPONENTS_PER_ENDPOINT,
        svd_solver="randomized",
        random_state=RANDOM_SEED,
        whiten=False,
    )
    pca.fit(vectors[train_endpoint_indices])
    reduced = pca.transform(vectors).astype(np.float32)

    if reduced.shape != (
        len(vectors),
        PCA_COMPONENTS_PER_ENDPOINT,
    ):
        raise RuntimeError(
            f"Unexpected PCA matrix shape {reduced.shape}."
        )
    if not np.isfinite(reduced).all():
        raise RuntimeError("PCA produced non-finite values.")

    elapsed = time.perf_counter() - started
    return reduced, pca, elapsed


## 8. Common RF and SNN protocol

RF uses one identical three-candidate search budget across all benchmark
notebooks. The SNN below is the exact shared project architecture from the
provided SNN baseline notebook. No K-fold cross-validation is generated.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from sklearn.model_selection import ParameterSampler, train_test_split
from torch.utils.data import DataLoader, Dataset

if not torch.cuda.is_available():
    raise RuntimeError(
        "SNN baselines require a GPU. In Kaggle select 2x T4."
    )
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
if GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{GPU_NAME} has unsupported CUDA capability "
        f"sm_{GPU_CAPABILITY[0]}{GPU_CAPABILITY[1]}. "
        "Use a T4-class accelerator."
    )
SNN_DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(
    {
        "gpu": GPU_NAME,
        "capability": (
            f"sm_{GPU_CAPABILITY[0]}{GPU_CAPABILITY[1]}"
        ),
        "gpu_count": torch.cuda.device_count(),
        "note": (
            "Reference SNN is single-process and uses GPU 0; "
            "2x T4 is accepted but not data-parallel."
        ),
    }
)



def safe_json_value(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            str(key): safe_json_value(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [
            safe_json_value(item)
            for item in value
        ]
    if isinstance(value, np.generic):
        return value.item()
    return value

RF_SPACE = {
    "n_estimators": [120, 180, 240],
    "max_depth": [18, 24, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.10, 0.20],
    "bootstrap": [True],
    "max_samples": [0.65, 0.80],
}


def stable_seed(namespace: str) -> int:
    digest = hashlib.sha256(
        f"{RANDOM_SEED}|{namespace}".encode("utf-8")
    ).digest()
    return int.from_bytes(digest[:4], "little")


def stratified_cap_positions(
    available_positions: np.ndarray,
    cap: Optional[int],
    namespace: str,
) -> np.ndarray:
    positions = np.asarray(
        available_positions,
        dtype=np.int64,
    )
    if cap is None or len(positions) <= int(cap):
        return positions
    strata = y_all[positions].astype(str)
    selected, _ = train_test_split(
        positions,
        train_size=int(cap),
        random_state=stable_seed(namespace),
        stratify=strata,
    )
    return np.sort(selected.astype(np.int64))


def metrics_at_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    labels = np.asarray(labels, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    preds = (scores >= threshold).astype(np.int8)

    return {
        "precision": float(
            precision_score(
                labels,
                preds,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                labels,
                preds,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                labels,
                preds,
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(labels, preds)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(labels, preds)
        ),
        "mcc": float(
            matthews_corrcoef(labels, preds)
        ),
        "auc": float(
            roc_auc_score(labels, scores)
        ) if len(np.unique(labels)) == 2 else float("nan"),
        "average_precision": float(
            average_precision_score(labels, scores)
        ) if len(np.unique(labels)) == 2 else float("nan"),
    }


def best_f1_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
) -> float:
    # Same 501-quantile candidate rule as codexglue-snn-baselines,
    # implemented with sorted cumulative counts for speed.
    labels = np.asarray(labels, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    candidates = np.unique(
        np.quantile(
            scores,
            np.linspace(0.0, 1.0, 501),
        )
    )
    candidates = np.unique(
        np.concatenate(
            [
                candidates,
                np.asarray([0.5], dtype=np.float32),
            ]
        )
    )

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    prefix_pos = np.concatenate(
        [[0], np.cumsum(sorted_labels, dtype=np.int64)]
    )
    total_pos = int(prefix_pos[-1])
    total = len(labels)

    best_threshold = 0.5
    best_f1 = -1.0
    best_acc = -1.0

    for threshold in candidates:
        first_positive_pred = int(
            np.searchsorted(
                sorted_scores,
                threshold,
                side="left",
            )
        )
        predicted_pos = total - first_positive_pred
        tp = total_pos - int(
            prefix_pos[first_positive_pred]
        )
        fp = predicted_pos - tp
        fn = total_pos - tp
        tn = total - tp - fp - fn

        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        f1 = (
            2.0 * precision * recall
            / max(1e-12, precision + recall)
        )
        acc = (tp + tn) / max(1, total)
        if (f1, acc) > (best_f1, best_acc):
            best_threshold = float(threshold)
            best_f1 = f1
            best_acc = acc

    return best_threshold


def batched_cosine_scores(
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    positions = np.asarray(positions, dtype=np.int64)
    scores = np.empty(len(positions), dtype=np.float64)
    for start in range(
        0,
        len(positions),
        PREDICTION_BATCH_SIZE,
    ):
        end = min(
            start + PREDICTION_BATCH_SIZE,
            len(positions),
        )
        local = positions[start:end]
        left = vectors[left_indices[local]]
        right = vectors[right_indices[local]]
        numerator = np.einsum(
            "ij,ij->i",
            left,
            right,
            optimize=True,
        )
        denominator = (
            np.linalg.norm(left, axis=1)
            * np.linalg.norm(right, axis=1)
        )
        scores[start:end] = (
            numerator
            / np.maximum(denominator, 1e-12)
        )
    return scores


class PairIndexDataset(Dataset):
    def __init__(
        self,
        positions: np.ndarray,
        left_indices: np.ndarray,
        right_indices: np.ndarray,
        labels: np.ndarray,
    ):
        positions = np.asarray(positions, dtype=np.int64)
        self.left_idx = torch.from_numpy(
            left_indices[positions].astype(
                np.int64,
                copy=False,
            )
        )
        self.right_idx = torch.from_numpy(
            right_indices[positions].astype(
                np.int64,
                copy=False,
            )
        )
        self.labels = torch.from_numpy(
            labels[positions].astype(
                np.float32,
                copy=False,
            )
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.left_idx[idx],
            self.right_idx[idx],
            self.labels[idx],
        )


class SiameseSpectralNet(nn.Module):
    """Exact SNN architecture used by codexglue-snn-baselines.ipynb."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        embed_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )
        self.head = nn.Sequential(
            nn.Linear(
                embed_dim * 2 + 2,
                hidden_dim,
            ),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        left_vecs,
        right_vecs,
    ):
        left = F.normalize(
            self.encoder(left_vecs),
            dim=-1,
        )
        right = F.normalize(
            self.encoder(right_vecs),
            dim=-1,
        )
        diff = torch.abs(left - right)
        prod = left * right
        cosine = prod.sum(
            dim=-1,
            keepdim=True,
        )
        l2 = torch.linalg.vector_norm(
            left - right,
            dim=-1,
            keepdim=True,
        )
        return self.head(
            torch.cat(
                [diff, prod, cosine, l2],
                dim=-1,
            )
        ).squeeze(-1)


def make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=(
                SNN_DEVICE == "cuda"
                and enabled
            ),
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=(
                SNN_DEVICE == "cuda"
                and enabled
            )
        )


def autocast_context(enabled: bool):
    if SNN_DEVICE == "cuda" and enabled:
        return torch.amp.autocast("cuda")
    return nullcontext()


In [ ]:
split_values = pairs["split"].astype(str).to_numpy()
train_positions = np.flatnonzero(
    split_values == "train"
)
validation_positions = np.flatnonzero(
    split_values == "validation"
)
test_positions = np.flatnonzero(
    split_values == "test"
)
y_all = pairs["label"].to_numpy(dtype=np.int8)

if QUICK_TEST:
    quick_positions = []
    for split_name, positions in (
        ("train", train_positions),
        ("validation", validation_positions),
        ("test", test_positions),
    ):
        local = pairs.iloc[positions].copy()
        local["_position"] = positions
        local["_quick_hash"] = local["pair_id"].map(
            lambda value: hashlib.sha256(
                f"{RANDOM_SEED}|{split_name}|{value}".encode(
                    "utf-8"
                )
            ).hexdigest()
        )

        grouping = ["label"]
        if (
            SUBGROUP_COLUMN in local.columns
            and local[SUBGROUP_COLUMN].notna().any()
        ):
            grouping = [SUBGROUP_COLUMN, "label"]

        selected = (
            local.sort_values("_quick_hash")
            .groupby(
                grouping,
                observed=True,
                group_keys=False,
            )
            .head(QUICK_TEST_ROWS_PER_STRATUM)
        )
        quick_positions.extend(
            selected["_position"].astype(int).tolist()
        )

    quick_set = set(quick_positions)
    train_positions = np.asarray(
        [
            p for p in train_positions
            if int(p) in quick_set
        ],
        dtype=np.int64,
    )
    validation_positions = np.asarray(
        [
            p for p in validation_positions
            if int(p) in quick_set
        ],
        dtype=np.int64,
    )
    test_positions = np.asarray(
        [
            p for p in test_positions
            if int(p) in quick_set
        ],
        dtype=np.int64,
    )

rf_train_positions = stratified_cap_positions(
    train_positions,
    RF_TRAIN_ROW_CAP,
    "rf_common_train_sample",
)
snn_train_positions = stratified_cap_positions(
    train_positions,
    SNN_TRAIN_ROW_CAP,
    "snn_common_train_sample",
)

print(
    "train / validation / test:",
    len(train_positions),
    len(validation_positions),
    len(test_positions),
)
print(
    "RF fitting rows:",
    len(rf_train_positions),
)
print(
    "SNN fitting rows:",
    len(snn_train_positions),
)
print(
    "Cross-validation: NOT USED; fixed validation split only."
)


In [ ]:
def prediction_metrics_frame(
    embedding_model: str,
    method_name: str,
    prediction_frame: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    labels = prediction_frame["label"].to_numpy(
        dtype=np.int8
    )
    preds = prediction_frame["prediction"].to_numpy(
        dtype=np.int8
    )
    scores = prediction_frame["score"].to_numpy(
        dtype=np.float64
    )

    overall = {
        "embedding_model": embedding_model,
        "method": method_name,
        "split": "test",
        "scope": "overall",
        "subset": "all",
        "rows": len(prediction_frame),
        "positive_rows": int((labels == 1).sum()),
        "negative_rows": int((labels == 0).sum()),
        **metrics_at_threshold(
            labels,
            scores,
            0.5,
        ),
    }
    # metrics_at_threshold above thresholds scores again. For learned
    # methods score may not use 0.5 as final decision, so replace
    # classification metrics with the actual stored predictions.
    overall.update(
        {
            "precision": float(
                precision_score(
                    labels,
                    preds,
                    zero_division=0,
                )
            ),
            "recall": float(
                recall_score(
                    labels,
                    preds,
                    zero_division=0,
                )
            ),
            "f1": float(
                f1_score(
                    labels,
                    preds,
                    zero_division=0,
                )
            ),
            "accuracy": float(
                accuracy_score(labels, preds)
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    labels,
                    preds,
                )
            ),
            "mcc": float(
                matthews_corrcoef(
                    labels,
                    preds,
                )
            ),
        }
    )
    rows.append(overall)

    if (
        SUBGROUP_COLUMN in prediction_frame.columns
        and prediction_frame[
            SUBGROUP_COLUMN
        ].notna().any()
    ):
        for subset_value, group in prediction_frame.groupby(
            SUBGROUP_COLUMN,
            dropna=False,
        ):
            y = group["label"].to_numpy(dtype=np.int8)
            p = group["prediction"].to_numpy(dtype=np.int8)
            s = group["score"].to_numpy(dtype=np.float64)
            row = {
                "embedding_model": embedding_model,
                "method": method_name,
                "split": "test",
                "scope": SUBGROUP_COLUMN,
                "subset": str(subset_value),
                "rows": len(group),
                "positive_rows": int((y == 1).sum()),
                "negative_rows": int((y == 0).sum()),
                "precision": float(
                    precision_score(
                        y,
                        p,
                        zero_division=0,
                    )
                ),
                "recall": float(
                    recall_score(
                        y,
                        p,
                        zero_division=0,
                    )
                ),
                "f1": float(
                    f1_score(
                        y,
                        p,
                        zero_division=0,
                    )
                ),
                "accuracy": float(
                    accuracy_score(y, p)
                ),
                "balanced_accuracy": float(
                    balanced_accuracy_score(y, p)
                ),
                "mcc": float(
                    matthews_corrcoef(y, p)
                ),
                "auc": (
                    float(roc_auc_score(y, s))
                    if len(np.unique(y)) == 2
                    else float("nan")
                ),
                "average_precision": (
                    float(
                        average_precision_score(y, s)
                    )
                    if len(np.unique(y)) == 2
                    else float("nan")
                ),
            }
            rows.append(row)

    return pd.DataFrame(rows)


def make_prediction_frame(
    positions: np.ndarray,
    predictions: np.ndarray,
    scores: np.ndarray,
) -> pd.DataFrame:
    columns = [
        column
        for column in (
            "pair_id",
            "label",
            "split",
            SUBGROUP_COLUMN,
        )
        if column in pairs.columns
    ]
    frame = pairs.iloc[positions][columns].copy()
    frame = frame.reset_index(drop=True)
    frame["prediction"] = np.asarray(
        predictions,
        dtype=np.int8,
    )
    frame["score"] = np.asarray(
        scores,
        dtype=np.float64,
    )
    return frame


def run_no_train(
    embedding_model: str,
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    started = time.perf_counter()
    valid_scores = batched_cosine_scores(
        vectors,
        left_indices,
        right_indices,
        validation_positions,
    )
    threshold = best_f1_threshold(
        y_all[validation_positions],
        valid_scores,
    )
    valid_metrics = metrics_at_threshold(
        y_all[validation_positions],
        valid_scores,
        threshold,
    )

    test_scores = batched_cosine_scores(
        vectors,
        left_indices,
        right_indices,
        test_positions,
    )
    test_predictions = (
        test_scores >= threshold
    ).astype(np.int8)

    prediction_frame = make_prediction_frame(
        test_positions,
        test_predictions,
        test_scores,
    )
    method = (
        f"{MODEL_DISPLAY_NAME[embedding_model]} + No Train"
    )
    metrics_frame = prediction_metrics_frame(
        embedding_model,
        method,
        prediction_frame,
    )
    elapsed = time.perf_counter() - started

    details = {
        "method": method,
        "variant": "No Train",
        "validation_threshold": float(threshold),
        "validation_f1": float(valid_metrics["f1"]),
        "trainable_parameters": 0,
        "params_display": "0",
        "device": "CPU",
        "model_compute_seconds": float(elapsed),
        "train_pairs": 0,
        "validation_pairs": len(validation_positions),
        "test_pairs": len(test_positions),
        "input_dimension_per_endpoint": int(
            vectors.shape[1]
        ),
        "pair_feature_dimension": int(
            vectors.shape[1] * 2
        ),
    }
    return details, metrics_frame, prediction_frame


In [ ]:
def rf_candidate_parameters() -> list[dict[str, Any]]:
    return list(
        ParameterSampler(
            RF_SPACE,
            n_iter=RF_SEARCH_ITERATIONS,
            random_state=RANDOM_SEED,
        )
    )


def rf_predict_batched(
    model,
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    positions: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    predictions = np.empty(
        len(positions),
        dtype=np.int8,
    )
    scores = np.empty(
        len(positions),
        dtype=np.float64,
    )
    for start in range(
        0,
        len(positions),
        PREDICTION_BATCH_SIZE,
    ):
        end = min(
            start + PREDICTION_BATCH_SIZE,
            len(positions),
        )
        local_positions = positions[start:end]
        X_batch = materialize_pair_features(
            vectors,
            left_indices,
            right_indices,
            local_positions,
        )
        predictions[start:end] = model.predict(
            X_batch
        ).astype(np.int8)
        scores[start:end] = model.predict_proba(
            X_batch
        )[:, 1]
        del X_batch
    return predictions, scores


def run_rf(
    embedding_model: str,
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    *,
    use_pca: bool,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    started = time.perf_counter()
    method = (
        f"{MODEL_DISPLAY_NAME[embedding_model]} + "
        + ("PCA + RF" if use_pca else "RF")
    )

    X_train = materialize_pair_features(
        vectors,
        left_indices,
        right_indices,
        rf_train_positions,
    )
    y_train = y_all[rf_train_positions]
    y_valid = y_all[validation_positions]

    search_rows = []
    candidates = rf_candidate_parameters()

    for candidate_index, parameters in enumerate(
        candidates
    ):
        candidate_started = time.perf_counter()
        model = RandomForestClassifier(
            n_jobs=N_JOBS,
            random_state=RANDOM_SEED,
            **parameters,
        )
        model.fit(X_train, y_train)
        valid_prediction, valid_scores = rf_predict_batched(
            model,
            vectors,
            left_indices,
            right_indices,
            validation_positions,
        )
        valid_metrics = {
            "precision": float(
                precision_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0,
                )
            ),
            "recall": float(
                recall_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0,
                )
            ),
            "f1": float(
                f1_score(
                    y_valid,
                    valid_prediction,
                    zero_division=0,
                )
            ),
            "accuracy": float(
                accuracy_score(
                    y_valid,
                    valid_prediction,
                )
            ),
            "auc": float(
                roc_auc_score(
                    y_valid,
                    valid_scores,
                )
            ),
            "average_precision": float(
                average_precision_score(
                    y_valid,
                    valid_scores,
                )
            ),
        }
        search_rows.append(
            {
                "embedding_model": embedding_model,
                "method": method,
                "candidate_index": candidate_index,
                "parameters_json": json.dumps(
                    safe_json_value(parameters),
                    sort_keys=True,
                ),
                "fit_rows": len(rf_train_positions),
                "validation_rows": len(
                    validation_positions
                ),
                "elapsed_seconds": (
                    time.perf_counter()
                    - candidate_started
                ),
                **valid_metrics,
            }
        )
        del model, valid_prediction, valid_scores
        gc.collect()

    search = pd.DataFrame(search_rows).sort_values(
        [
            "f1",
            "auc",
            "accuracy",
            "candidate_index",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    ).reset_index(drop=True)
    best_index = int(
        search.loc[0, "candidate_index"]
    )
    best_parameters = candidates[best_index]

    # Refit the selected RF on the same train-only sample. Validation is
    # used only for selection, matching the SNN reference protocol.
    final_model = RandomForestClassifier(
        n_jobs=N_JOBS,
        random_state=RANDOM_SEED,
        **best_parameters,
    )
    final_model.fit(X_train, y_train)
    del X_train
    gc.collect()

    test_prediction, test_scores = rf_predict_batched(
        final_model,
        vectors,
        left_indices,
        right_indices,
        test_positions,
    )
    prediction_frame = make_prediction_frame(
        test_positions,
        test_prediction,
        test_scores,
    )
    metrics_frame = prediction_metrics_frame(
        embedding_model,
        method,
        prediction_frame,
    )
    elapsed = time.perf_counter() - started

    details = {
        "method": method,
        "variant": (
            "PCA + RF" if use_pca else "RF"
        ),
        "validation_f1": float(
            search.loc[0, "f1"]
        ),
        "validation_auc": float(
            search.loc[0, "auc"]
        ),
        "best_parameters": safe_json_value(
            best_parameters
        ),
        "trainable_parameters": None,
        "params_display": "-",
        "device": f"CPU (n_jobs={N_JOBS})",
        "model_compute_seconds": float(elapsed),
        "train_pairs": len(rf_train_positions),
        "validation_pairs": len(
            validation_positions
        ),
        "test_pairs": len(test_positions),
        "input_dimension_per_endpoint": int(
            vectors.shape[1]
        ),
        "pair_feature_dimension": int(
            vectors.shape[1] * 2
        ),
    }

    if SAVE_FITTED_MODELS:
        model_slug = MODEL_SLUG[embedding_model]
        suffix = "pca_rf" if use_pca else "rf"
        joblib.dump(
            final_model,
            OUTPUT_ROOT
            / f"{model_slug}__{suffix}__model.joblib",
            compress=3,
        )

    del final_model
    gc.collect()
    return details, metrics_frame, prediction_frame, search


In [ ]:
def snn_loader(
    positions: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    *,
    shuffle: bool,
) -> DataLoader:
    kwargs = {
        "batch_size": SNN_BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": 0,
        "pin_memory": True,
    }
    return DataLoader(
        PairIndexDataset(
            positions,
            left_indices,
            right_indices,
            y_all,
        ),
        **kwargs,
    )


@torch.inference_mode()
def snn_predict(
    model: nn.Module,
    code_tensor: torch.Tensor,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    labels_out = []
    scores_out = []
    for left_idx, right_idx, labels in loader:
        left_idx = left_idx.to(
            SNN_DEVICE,
            non_blocking=True,
        )
        right_idx = right_idx.to(
            SNN_DEVICE,
            non_blocking=True,
        )
        logits = model(
            code_tensor[left_idx],
            code_tensor[right_idx],
        )
        labels_out.append(labels.numpy())
        scores_out.append(
            torch.sigmoid(logits)
            .detach()
            .float()
            .cpu()
            .numpy()
        )
    return (
        np.concatenate(labels_out).astype(np.int8),
        np.concatenate(scores_out).astype(np.float64),
    )


def run_snn(
    embedding_model: str,
    vectors: np.ndarray,
    left_indices: np.ndarray,
    right_indices: np.ndarray,
    *,
    use_pca: bool,
) -> tuple[
    dict[str, Any],
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    started = time.perf_counter()
    method = (
        f"{MODEL_DISPLAY_NAME[embedding_model]} + "
        + ("PCA + SNN" if use_pca else "SNN")
    )

    seed = RANDOM_SEED
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    code_tensor = torch.from_numpy(
        np.ascontiguousarray(vectors)
    ).float().to(SNN_DEVICE)

    train_loader = snn_loader(
        snn_train_positions,
        left_indices,
        right_indices,
        shuffle=True,
    )
    valid_loader = snn_loader(
        validation_positions,
        left_indices,
        right_indices,
        shuffle=False,
    )
    test_loader = snn_loader(
        test_positions,
        left_indices,
        right_indices,
        shuffle=False,
    )

    input_dim = int(vectors.shape[1])
    model = SiameseSpectralNet(
        input_dim,
        SNN_HIDDEN_DIM,
        SNN_EMBED_DIM,
        SNN_DROPOUT,
    ).to(SNN_DEVICE)

    trainable_parameters = int(
        sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SNN_LEARNING_RATE,
        weight_decay=SNN_WEIGHT_DECAY,
    )
    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=SNN_SCHEDULER_FACTOR,
            patience=SNN_SCHEDULER_PATIENCE,
        )
    )

    if SNN_USE_POS_WEIGHT:
        y_train_local = y_all[snn_train_positions]
        positive = float(
            (y_train_local == 1).sum()
        )
        negative = float(
            (y_train_local == 0).sum()
        )
        pos_weight = torch.tensor(
            [negative / max(1.0, positive)],
            device=SNN_DEVICE,
        )
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )
    else:
        criterion = nn.BCEWithLogitsLoss()

    scaler = make_scaler(SNN_USE_AMP)
    best_state = None
    best_valid_f1 = -1.0
    best_threshold = 0.5
    best_epoch = 0
    stale_epochs = 0
    history_rows = []

    for epoch in range(
        1,
        SNN_MAX_EPOCHS + 1,
    ):
        model.train()
        total_loss = 0.0
        seen = 0

        for left_idx, right_idx, labels in train_loader:
            left_idx = left_idx.to(
                SNN_DEVICE,
                non_blocking=True,
            )
            right_idx = right_idx.to(
                SNN_DEVICE,
                non_blocking=True,
            )
            labels = labels.to(
                SNN_DEVICE,
                non_blocking=True,
            )
            optimizer.zero_grad(set_to_none=True)

            with autocast_context(SNN_USE_AMP):
                logits = model(
                    code_tensor[left_idx],
                    code_tensor[right_idx],
                )
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                SNN_GRAD_CLIP_NORM,
            )
            scaler.step(optimizer)
            scaler.update()

            batch_size = int(labels.numel())
            total_loss += (
                float(loss.detach().cpu())
                * batch_size
            )
            seen += batch_size

        train_loss = total_loss / max(1, seen)
        y_valid, valid_scores = snn_predict(
            model,
            code_tensor,
            valid_loader,
        )
        threshold = best_f1_threshold(
            y_valid,
            valid_scores,
        )
        valid_metrics = metrics_at_threshold(
            y_valid,
            valid_scores,
            threshold,
        )
        scheduler.step(valid_metrics["f1"])

        history_rows.append(
            {
                "embedding_model": embedding_model,
                "method": method,
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_precision": (
                    valid_metrics["precision"]
                ),
                "valid_recall": (
                    valid_metrics["recall"]
                ),
                "valid_f1": valid_metrics["f1"],
                "valid_accuracy": (
                    valid_metrics["accuracy"]
                ),
                "threshold": threshold,
                "lr": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"{method}: epoch={epoch:02d} "
            f"loss={train_loss:.4f} "
            f"valid_f1={valid_metrics['f1']:.4f} "
            f"thr={threshold:.4f}"
        )

        if valid_metrics["f1"] > best_valid_f1:
            best_valid_f1 = valid_metrics["f1"]
            best_threshold = threshold
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= SNN_PATIENCE:
                print(
                    f"{method}: early stopping at "
                    f"epoch {epoch}; best={best_epoch}"
                )
                break

    if best_state is None:
        raise RuntimeError(
            f"{method}: no valid SNN checkpoint."
        )
    model.load_state_dict(
        {
            key: value.to(SNN_DEVICE)
            for key, value in best_state.items()
        }
    )

    y_test, test_scores = snn_predict(
        model,
        code_tensor,
        test_loader,
    )
    if not np.array_equal(
        y_test,
        y_all[test_positions],
    ):
        raise RuntimeError(
            f"{method}: test label order changed."
        )
    test_prediction = (
        test_scores >= best_threshold
    ).astype(np.int8)

    prediction_frame = make_prediction_frame(
        test_positions,
        test_prediction,
        test_scores,
    )
    metrics_frame = prediction_metrics_frame(
        embedding_model,
        method,
        prediction_frame,
    )
    history = pd.DataFrame(history_rows)
    elapsed = time.perf_counter() - started

    details = {
        "method": method,
        "variant": (
            "PCA + SNN" if use_pca else "SNN"
        ),
        "best_epoch": int(best_epoch),
        "validation_threshold": float(
            best_threshold
        ),
        "validation_f1": float(
            best_valid_f1
        ),
        "trainable_parameters": (
            trainable_parameters
        ),
        "params_display": (
            f"{trainable_parameters / 1000:.1f}K"
            if trainable_parameters < 1_000_000
            else f"{trainable_parameters / 1_000_000:.2f}M"
        ),
        "device": f"{GPU_NAME} (cuda:0)",
        "model_compute_seconds": float(elapsed),
        "train_pairs": len(snn_train_positions),
        "validation_pairs": len(
            validation_positions
        ),
        "test_pairs": len(test_positions),
        "input_dimension_per_endpoint": input_dim,
        "pair_feature_dimension": (
            input_dim * 2
        ),
    }

    if SAVE_FITTED_MODELS:
        model_slug = MODEL_SLUG[embedding_model]
        suffix = "pca_snn" if use_pca else "snn"
        torch.save(
            {
                "state_dict": best_state,
                "input_dim": input_dim,
                "hidden_dim": SNN_HIDDEN_DIM,
                "embed_dim": SNN_EMBED_DIM,
                "dropout": SNN_DROPOUT,
                "threshold": best_threshold,
            },
            OUTPUT_ROOT
            / f"{model_slug}__{suffix}__model.pt",
        )

    del (
        model,
        optimizer,
        scheduler,
        code_tensor,
        train_loader,
        valid_loader,
        test_loader,
        best_state,
    )
    gc.collect()
    torch.cuda.empty_cache()

    return (
        details,
        metrics_frame,
        prediction_frame,
        history,
    )


## 9. Run the 20 Pretrained Baselines

Each of the four embedding models produces five paper rows. The same RF sample
positions are reused across embeddings and raw/PCA representations within a
benchmark. SNN uses the configured final-full training split.


In [ ]:
# Fail fast before loading embeddings if a required baseline implementation
# was not executed/defined in an earlier code cell.
_REQUIRED_BASELINE_CALLABLES = (
    "run_no_train",
    "rf_candidate_parameters",
    "rf_predict_batched",
    "run_rf",
    "snn_loader",
    "snn_predict",
    "run_snn",
)
_missing_baseline_callables = [
    name
    for name in _REQUIRED_BASELINE_CALLABLES
    if not callable(globals().get(name))
]
if _missing_baseline_callables:
    raise RuntimeError(
        "Baseline dependency preflight failed. Missing callables: "
        + ", ".join(_missing_baseline_callables)
        + ". Run the notebook from the top with Run All."
    )
print(
    "Baseline dependency preflight: PASS —",
    ", ".join(_REQUIRED_BASELINE_CALLABLES),
)

all_details = []
all_metric_frames = []
embedding_runtime_rows = []

for embedding_model in EMBEDDING_MODELS_TO_RUN:
    print("\n" + "=" * 90)
    print("EMBEDDING:", embedding_model)
    print("=" * 90)

    (
        raw_vectors,
        embedding_ids,
        row_by_id,
        embedding_load_seconds,
    ) = load_endpoint_embeddings(
        embedding_model
    )
    left_indices, right_indices = (
        pair_endpoint_indices(row_by_id)
    )
    del row_by_id

    (
        pca_vectors,
        pca_model,
        pca_seconds,
    ) = fit_pca32_from_train_endpoints(
        raw_vectors,
        left_indices,
        right_indices,
        train_positions,
    )
    explained_variance = float(
        pca_model.explained_variance_ratio_.sum()
    )

    embedding_runtime_rows.append(
        {
            "embedding_model": embedding_model,
            "embedding_load_seconds": (
                embedding_load_seconds
            ),
            "pca_fit_transform_seconds": pca_seconds,
            "pca_components_per_endpoint": (
                PCA_COMPONENTS_PER_ENDPOINT
            ),
            "pca_explained_variance_ratio_sum": (
                explained_variance
            ),
        }
    )

    runs = [
        (
            "no_train",
            lambda: run_no_train(
                embedding_model,
                raw_vectors,
                left_indices,
                right_indices,
            ),
            False,
        ),
        (
            "rf",
            lambda: run_rf(
                embedding_model,
                raw_vectors,
                left_indices,
                right_indices,
                use_pca=False,
            ),
            False,
        ),
        (
            "snn",
            lambda: run_snn(
                embedding_model,
                raw_vectors,
                left_indices,
                right_indices,
                use_pca=False,
            ),
            False,
        ),
        (
            "pca_rf",
            lambda: run_rf(
                embedding_model,
                pca_vectors,
                left_indices,
                right_indices,
                use_pca=True,
            ),
            True,
        ),
        (
            "pca_snn",
            lambda: run_snn(
                embedding_model,
                pca_vectors,
                left_indices,
                right_indices,
                use_pca=True,
            ),
            True,
        ),
    ]

    for method_slug, runner, uses_pca in runs:
        prefix = (
            f"{MODEL_SLUG[embedding_model]}"
            f"__{method_slug}"
        )
        success_path = (
            OUTPUT_ROOT
            / f"{prefix}__SUCCESS.json"
        )

        print("\n---", prefix, "---")
        result = runner()

        if method_slug == "no_train":
            (
                details,
                metrics_frame,
                prediction_frame,
            ) = result
            extra_frame = pd.DataFrame(
                [
                    {
                        "threshold": details[
                            "validation_threshold"
                        ],
                        "validation_f1": details[
                            "validation_f1"
                        ],
                    }
                ]
            )
            extra_name = "threshold_selection"
        else:
            (
                details,
                metrics_frame,
                prediction_frame,
                extra_frame,
            ) = result
            extra_name = (
                "training_history"
                if "snn" in method_slug
                else "validation_search"
            )

        standalone_overhead = (
            embedding_load_seconds
            + (pca_seconds if uses_pca else 0.0)
        )
        details.update(
            {
                "experiment_version": (
                    EXPERIMENT_VERSION
                ),
                "source_benchmark": (
                    SOURCE_BENCHMARK_NAME
                ),
                "paper_dataset": PAPER_DATASET_NAME,
                "embedding_model": embedding_model,
                "embedding_load_seconds": (
                    embedding_load_seconds
                ),
                "pca_fit_transform_seconds": (
                    pca_seconds
                    if uses_pca
                    else 0.0
                ),
                "pca_explained_variance_ratio_sum": (
                    explained_variance
                    if uses_pca
                    else None
                ),
                "reported_runtime_seconds": float(
                    details[
                        "model_compute_seconds"
                    ]
                    + standalone_overhead
                ),
                "reported_runtime_minutes": float(
                    (
                        details[
                            "model_compute_seconds"
                        ]
                        + standalone_overhead
                    )
                    / 60.0
                ),
                "runtime_scope": (
                    "load precomputed embeddings + "
                    "PCA when applicable + model "
                    "selection/training + complete "
                    "validation/test evaluation; "
                    "pretrained encoder extraction excluded"
                ),
                "params_scope": (
                    "trainable downstream parameters only; "
                    "frozen pretrained encoder excluded"
                ),
                "cross_validation_used": False,
                "test_used_for_selection": False,
                "status": "complete",
                "completed_at_utc": (
                    datetime.now(
                        timezone.utc
                    ).isoformat()
                ),
            }
        )

        metrics_frame.to_csv(
            OUTPUT_ROOT
            / f"{prefix}__metrics.csv",
            index=False,
        )
        extra_frame.to_csv(
            OUTPUT_ROOT
            / f"{prefix}__{extra_name}.csv",
            index=False,
        )
        if SAVE_TEST_PREDICTIONS:
            prediction_frame.to_parquet(
                OUTPUT_ROOT
                / f"{prefix}__test_predictions.parquet",
                index=False,
                compression="zstd",
            )

        success_path.write_text(
            json.dumps(
                safe_json_value(details),
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        all_details.append(details)
        all_metric_frames.append(metrics_frame)
        display(
            metrics_frame[
                metrics_frame["scope"] == "overall"
            ]
        )

        del metrics_frame, prediction_frame, extra_frame
        gc.collect()

    del (
        raw_vectors,
        pca_vectors,
        pca_model,
        embedding_ids,
        left_indices,
        right_indices,
    )
    gc.collect()
    torch.cuda.empty_cache()

print(
    "Completed baseline methods:",
    len(all_details),
)


In [ ]:
expected_methods = (
    len(EMBEDDING_MODELS_TO_RUN)
    * len(BASELINE_VARIANTS)
)
if len(all_details) != expected_methods:
    raise RuntimeError(
        f"Expected {expected_methods} completed methods, "
        f"found {len(all_details)}."
    )

all_metrics = pd.concat(
    all_metric_frames,
    ignore_index=True,
)
all_metrics.to_csv(
    OUTPUT_ROOT
    / "pretrained_baselines_all_metrics.csv",
    index=False,
)

overall = all_metrics[
    (all_metrics["scope"] == "overall")
    & (all_metrics["split"] == "test")
].copy()

method_order = []
for model in EMBEDDING_MODELS_TO_RUN:
    display_name = MODEL_DISPLAY_NAME[model]
    method_order.extend(
        [
            f"{display_name} + No Train",
            f"{display_name} + RF",
            f"{display_name} + SNN",
            f"{display_name} + PCA + RF",
            f"{display_name} + PCA + SNN",
        ]
    )

overall["method_order"] = pd.Categorical(
    overall["method"],
    categories=method_order,
    ordered=True,
)
overall = overall.sort_values(
    "method_order"
).drop(columns="method_order")

paper_metrics = overall[
    [
        "method",
        "precision",
        "recall",
        "f1",
        "accuracy",
    ]
].rename(
    columns={
        "method": "Method",
        "precision": "P",
        "recall": "R",
        "f1": "F1",
        "accuracy": "Acc",
    }
)
paper_metrics.insert(
    0,
    "Dataset",
    PAPER_DATASET_NAME,
)
paper_metrics.to_csv(
    OUTPUT_ROOT
    / "pretrained_baselines_paper_metrics.csv",
    index=False,
    float_format="%.6f",
)

details_frame = pd.DataFrame(all_details)
efficiency = details_frame[
    [
        "method",
        "reported_runtime_minutes",
        "params_display",
        "trainable_parameters",
        "device",
        "train_pairs",
        "validation_pairs",
        "test_pairs",
        "input_dimension_per_endpoint",
        "pair_feature_dimension",
        "embedding_load_seconds",
        "pca_fit_transform_seconds",
        "runtime_scope",
        "params_scope",
    ]
].rename(
    columns={
        "method": "Method",
        "reported_runtime_minutes": "Time (min)",
        "params_display": "Params",
        "device": "Device",
    }
)
efficiency.insert(
    0,
    "Dataset",
    PAPER_DATASET_NAME,
)
efficiency.to_csv(
    OUTPUT_ROOT
    / "pretrained_baselines_efficiency.csv",
    index=False,
    float_format="%.6f",
)

pd.DataFrame(
    embedding_runtime_rows
).to_csv(
    OUTPUT_ROOT
    / "embedding_and_pca_runtime.csv",
    index=False,
)

if SUBGROUP_COLUMN in all_metrics["scope"].unique():
    subgroup_metrics = all_metrics[
        all_metrics["scope"] == SUBGROUP_COLUMN
    ].copy()
    subgroup_metrics.to_csv(
        OUTPUT_ROOT
        / "pretrained_baselines_subgroup_metrics.csv",
        index=False,
    )

# Paper-ready dataset-local LaTeX fragment.
latex_lines = [
    (
        f"% {PAPER_DATASET_NAME} pretrained-baseline "
        "values: Method & P & R & F1 & Acc"
    )
]
for row in paper_metrics.itertuples(index=False):
    latex_lines.append(
        f"% {row.Method}: "
        f"{row.P:.4f} & {row.R:.4f} & "
        f"{row.F1:.4f} & {row.Acc:.4f}"
    )
(
    OUTPUT_ROOT
    / "pretrained_baselines_latex_values.tex"
).write_text(
    "\n".join(latex_lines) + "\n",
    encoding="utf-8",
)

print("\nPaper metrics:")
display(paper_metrics)
print("\nEfficiency table (para.png convention):")
display(
    efficiency[
        [
            "Dataset",
            "Method",
            "Time (min)",
            "Params",
            "Device",
        ]
    ]
)


In [ ]:
# Sanity-check the parameter convention against the reference SNN architecture.
def expected_snn_parameter_count(
    input_dim: int,
) -> int:
    h = SNN_HIDDEN_DIM
    e = SNN_EMBED_DIM
    total = 0
    total += input_dim * h + h
    total += 2 * h
    total += h * h + h
    total += 2 * h
    total += h * e + e
    total += (2 * e + 2) * h + h
    total += 2 * h
    total += h + 1
    return int(total)


expected_raw_snn_params = (
    expected_snn_parameter_count(
        FEATURE_DIMENSION
    )
)
expected_pca_snn_params = (
    expected_snn_parameter_count(
        PCA_COMPONENTS_PER_ENDPOINT
    )
)

observed_raw = set(
    details_frame.loc[
        details_frame["variant"] == "SNN",
        "trainable_parameters",
    ].dropna().astype(int)
)
observed_pca = set(
    details_frame.loc[
        details_frame["variant"] == "PCA + SNN",
        "trainable_parameters",
    ].dropna().astype(int)
)

if observed_raw != {expected_raw_snn_params}:
    raise RuntimeError(
        "Raw SNN parameter count does not match the "
        "reference architecture."
    )
if observed_pca != {expected_pca_snn_params}:
    raise RuntimeError(
        "PCA SNN parameter count does not match the "
        "reference architecture."
    )
if not (
    details_frame.loc[
        details_frame["variant"].str.startswith(
            "PCA"
        ),
        "pair_feature_dimension",
    ]
    .astype(int)
    .eq(PCA_PAIR_FEATURE_DIMENSION)
    .all()
):
    raise RuntimeError(
        "A PCA baseline does not use 32 + 32 = 64 "
        "pair features."
    )

parameter_report = {
    "reference_para_png_convention": {
        "No Train": "0 trainable parameters",
        "RF": "- (tree model; no fixed neural parameter count)",
        "SNN": (
            "trainable neural parameters only"
        ),
    },
    "reference_snn_recipe": (
        "SiameseSpectralNet from "
        "codexglue-snn-baselines.ipynb"
    ),
    "raw_embedding_endpoint_dim": FEATURE_DIMENSION,
    "raw_pair_dim": PAIR_FEATURE_DIMENSION,
    "raw_snn_trainable_parameters": (
        expected_raw_snn_params
    ),
    "pca_endpoint_dim": PCA_COMPONENTS_PER_ENDPOINT,
    "pca_pair_dim": PCA_PAIR_FEATURE_DIMENSION,
    "pca_snn_trainable_parameters": (
        expected_pca_snn_params
    ),
    "pretrained_encoder_parameters_counted": False,
    "reason": (
        "The experiment consumes frozen precomputed embeddings; "
        "Params follows the downstream-trainable convention "
        "used by para.png."
    ),
}
(
    OUTPUT_ROOT
    / "parameter_count_convention.json"
).write_text(
    json.dumps(
        parameter_report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print(
    "Raw SNN trainable parameters:",
    f"{expected_raw_snn_params:,}",
)
print(
    "PCA+SNN trainable parameters:",
    f"{expected_pca_snn_params:,}",
)
print(
    "PCA pair input:",
    f"{PCA_COMPONENTS_PER_ENDPOINT} + "
    f"{PCA_COMPONENTS_PER_ENDPOINT} = "
    f"{PCA_PAIR_FEATURE_DIMENSION}",
)


## 10. Final paper reports in the saved notebook output

The final output contains the compact performance table (`P`, `R`, `F1`,
`Acc`), efficiency table, all method metrics and search histories, optional
test predictions, and protocol manifests. Use **Save Version** to retain them;
no separate Kaggle Dataset is created.


In [ ]:
def owned_dataset_exists(
    handle: str,
) -> bool:
    owner, slug = handle.split("/", 1)
    if (
        owner.casefold()
        != KAGGLE_USERNAME.casefold()
    ):
        raise RuntimeError(
            "Output Dataset owner does not match "
            "KAGGLE_USERNAME."
        )

    result = run_command(
        [
            "kaggle",
            "datasets",
            "list",
            "--mine",
            "--search",
            slug,
            "--page",
            "1",
            "--csv",
        ],
        quiet=True,
        timeout=180,
    )
    rows = list(
        csv.reader(
            io.StringIO(
                (result.stdout or "").strip()
            )
        )
    )
    if not rows:
        return False

    header = [
        column.strip().casefold()
        for column in rows[0]
    ]
    ref_index = (
        header.index("ref")
        if "ref" in header
        else 0
    )
    references = {
        row[ref_index].strip().casefold()
        for row in rows[1:]
        if (
            len(row) > ref_index
            and "/" in row[ref_index]
        )
    }
    return (
        handle.casefold()
        in references
    )


def wait_for_dataset_ready(
    handle: str,
) -> str:
    last = ""
    for attempt in range(1, 31):
        result = run_command(
            [
                "kaggle",
                "datasets",
                "status",
                handle,
            ],
            check=False,
            quiet=True,
            timeout=120,
        )
        last = combined_output(result)
        lowered = last.casefold()
        if (
            result.returncode == 0
            and "ready" in lowered
        ):
            return last
        if any(
            marker in lowered
            for marker in ("failed", "error")
        ):
            raise RuntimeError(
                "Kaggle reported a failed version:\n"
                + last
            )
        if attempt < 30:
            time.sleep(10)
    raise RuntimeError(
        "Dataset version did not become ready:\n"
        + last
    )


def write_dataset_metadata() -> None:
    readme = f"""# {OUTPUT_DATASET_TITLE} — Pretrained Baselines

Experiment version: `{EXPERIMENT_VERSION}`

Source benchmark: `{SOURCE_BENCHMARK_NAME}`  
Paper dataset column: `{PAPER_DATASET_NAME}`

This version contains the paper's Pretrained Baselines:

- CodeBERT / GraphCodeBERT / UniXcoder / CodeT5
- No Train
- RF
- SNN
- PCA + RF
- PCA + SNN

PCA is fit using training endpoints only and reduces every endpoint from
768 to 32 components, so every PCA pair has 64 features.

The SNN is the same SiameseSpectralNet architecture and training recipe used
by `codexglue-snn-baselines.ipynb`: hidden=256, embed=128, dropout=0.10,
AdamW(lr=1e-3, weight_decay=1e-4), batch=8192, max epochs=10,
patience=3, ReduceLROnPlateau, BCEWithLogitsLoss, AMP, gradient clipping=5,
and validation-F1 threshold selection.

No cross-validation is used. The stored V3 validation split is used for
selection and the test split is evaluated only after selection.

For the efficiency table, `Params` follows the `para.png` convention:
No Train=0, RF='-', and SNN=trainable downstream neural parameters.
Frozen pretrained encoders are not included because this experiment consumes
precomputed embeddings. Runtime excludes original transformer embedding
extraction but includes loading those embeddings, PCA when applicable,
selection/training, and complete validation/test evaluation.
"""
    (
        OUTPUT_ROOT / "README.md"
    ).write_text(
        readme,
        encoding="utf-8",
    )

    metadata = {
        "title": OUTPUT_DATASET_TITLE,
        "id": OUTPUT_DATASET_HANDLE,
        "licenses": [{"name": "other"}],
        "subtitle": OUTPUT_DATASET_SUBTITLE,
        "description": readme,
    }
    (
        OUTPUT_ROOT
        / "dataset-metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )


def publish_output_dataset() -> dict[str, Any]:
    if QUICK_TEST:
        return {
            "status": "not_uploaded",
            "reason": "Quick test is never published.",
        }
    if not PUBLISH_TO_KAGGLE:
        return {"status": "disabled"}

    exists = owned_dataset_exists(
        OUTPUT_DATASET_HANDLE
    )
    if REQUIRE_EXISTING_OUTPUT_DATASET and not exists:
        raise RuntimeError(
            "The configured output Dataset does not exist: "
            + OUTPUT_DATASET_HANDLE
        )

    if exists:
        command = [
            "kaggle", "datasets", "version",
            "--path", str(OUTPUT_ROOT),
            "--message", OUTPUT_VERSION_NOTES + "; "
            + datetime.now(timezone.utc).isoformat(),
            "--keep-tabular", "--dir-mode", "skip",
        ]
        action = "version_private_existing"
    else:
        command = [
            "kaggle", "datasets", "create",
            "--path", str(OUTPUT_ROOT),
            "--keep-tabular", "--dir-mode", "skip",
        ]
        action = "create_private_new"

    run_command(command, timeout=14_400)
    remote_status = wait_for_dataset_ready(
        OUTPUT_DATASET_HANDLE
    )
    return {
        "status": "uploaded",
        "action": action,
        "handle": OUTPUT_DATASET_HANDLE,
        "visibility": "private",
        "remote_status": remote_status,
        "url": (
            "https://www.kaggle.com/datasets/"
            + OUTPUT_DATASET_HANDLE
        ),
    }


In [ ]:
expected_methods = (
    len(EMBEDDING_MODELS_TO_RUN)
    * len(BASELINE_VARIANTS)
)
success_files = sorted(
    OUTPUT_ROOT.glob("*__SUCCESS.json")
)

checks = {
    "all_20_methods_complete": (
        len(success_files)
        == expected_methods
        == len(all_details)
    ),
    "paper_metrics_have_20_rows": (
        len(paper_metrics)
        == expected_methods
    ),
    "efficiency_has_20_rows": (
        len(efficiency)
        == expected_methods
    ),
    "all_test_rows_complete": (
        overall["rows"]
        .astype(int)
        .eq(len(test_positions))
        .all()
    ),
    "pca_is_32_per_endpoint": (
        PCA_COMPONENTS_PER_ENDPOINT == 32
    ),
    "pca_pair_dimension_is_64": (
        PCA_PAIR_FEATURE_DIMENSION == 64
    ),
    "raw_pair_dimension_is_1536": (
        PAIR_FEATURE_DIMENSION == 1536
    ),
    "rf_search_iterations_are_uniform": (
        RF_SEARCH_ITERATIONS == 3
    ),
    "snn_recipe_matches_reference": (
        SNN_BATCH_SIZE == 8192
        and SNN_HIDDEN_DIM == 256
        and SNN_EMBED_DIM == 128
        and abs(SNN_DROPOUT - 0.10) < 1e-12
        and abs(SNN_LEARNING_RATE - 1e-3) < 1e-12
        and abs(SNN_WEIGHT_DECAY - 1e-4) < 1e-12
        and SNN_MAX_EPOCHS == 10
        and SNN_PATIENCE == 3
        and SNN_USE_POS_WEIGHT is False
    ),
    "cross_validation_not_used": True,
    "test_not_used_for_selection": True,
    "saved_notebook_output_mode": (not PUBLISH_TO_KAGGLE),
}

FINAL_VALIDATION = {
    "valid": all(checks.values()),
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "experiment_version": EXPERIMENT_VERSION,
    "source_benchmark": SOURCE_BENCHMARK_NAME,
    "paper_dataset": PAPER_DATASET_NAME,
    "source_data_format": SOURCE_DATA_FORMAT,
    "embedding_pipeline_version": (
        EXPECTED_EMBEDDING_PIPELINE_VERSION
    ),
    "methods": method_order,
    "rf_search_iterations": (
        RF_SEARCH_ITERATIONS
    ),
    "rf_train_row_cap": RF_TRAIN_ROW_CAP,
    "snn_train_row_cap": SNN_TRAIN_ROW_CAP,
    "snn_max_epochs": SNN_MAX_EPOCHS,
    "snn_patience": SNN_PATIENCE,
    "raw_endpoint_dim": FEATURE_DIMENSION,
    "raw_pair_dim": PAIR_FEATURE_DIMENSION,
    "pca_endpoint_dim": (
        PCA_COMPONENTS_PER_ENDPOINT
    ),
    "pca_pair_dim": (
        PCA_PAIR_FEATURE_DIMENSION
    ),
    "generated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

(
    OUTPUT_ROOT
    / "pretrained_baselines_final_validation.json"
).write_text(
    json.dumps(
        FINAL_VALIDATION,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

manifest = {
    **FINAL_VALIDATION,
    "source_benchmark_artifacts": "bundled_with_embedding_notebook_output",
    "source_embedding_notebook": SOURCE_EMBEDDING_NOTEBOOK,
    "embedding_input_method": EMBEDDING_INPUT_METHOD,
    "output_dataset": None,
    "output_directory": str(OUTPUT_ROOT),
    "publication_mode": "saved_notebook_output",
    "source_note": SOURCE_NOTE,
    "runtime_reporting_note": (
        "Reported method time includes loading precomputed "
        "embeddings, PCA when applicable, model selection/"
        "training, and full validation/test evaluation. "
        "Transformer embedding extraction is excluded."
    ),
    "parameter_reporting_note": (
        "Params follows para.png: No Train=0, RF='-', "
        "SNN=trainable downstream neural parameters. "
        "Frozen pretrained encoders are excluded."
    ),
}
(
    OUTPUT_ROOT
    / "pretrained_baselines_manifest.json"
).write_text(
    json.dumps(
        safe_json_value(manifest),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

write_dataset_metadata()

if not FINAL_VALIDATION["valid"]:
    raise RuntimeError(
        "Final pretrained-baseline validation failed."
    )

publication_report = (
    publish_output_dataset()
)
print(
    json.dumps(
        publication_report,
        indent=2,
    )
)
print(
    "\nPretrained-baseline experiment complete and valid."
)
print(
    "Paper metrics:",
    OUTPUT_ROOT
    / "pretrained_baselines_paper_metrics.csv",
)
print(
    "Efficiency metrics:",
    OUTPUT_ROOT
    / "pretrained_baselines_efficiency.csv",
)
